In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/dimaspashaakrilian/dsc-itb/Data_Klaim.csv
/kaggle/input/datasets/dimaspashaakrilian/dsc-itb/sample_submission.csv
/kaggle/input/datasets/dimaspashaakrilian/dsc-itb/Data_Polis.csv


# LOAD DATA
1. Membaca tiga file CSV (Klaim, Polis, Sample).
2. Mengecek ukuran dataset (jumlah baris & kolom).
3. Melihat struktur dan tipe data setiap kolom (info()).
4. Mengidentifikasi missing value pada masing-masing tabel.
5. Mengecek duplikasi Claim ID dan Nomor Polis.
6. Melihat statistik deskriptif nominal klaim (mean, median, max, dll).
7. Mengecek distribusi kategori penting (Status, Reimburse/Cashless, IP/OP, Plan Code).
8. Mengonversi kolom tanggal ke tipe datetime untuk analisis waktu.
9. Mengecek rentang tanggal kejadian dan pembayaran.
10. Mengidentifikasi bahwa tanggal pembayaran lebih panjang dari tanggal kejadian medis.

In [2]:
import numpy as np
import pandas as pd

BASE_PATH = "/kaggle/input/datasets/dimaspashaakrilian/dsc-itb/"

PATH_KLAIM  = BASE_PATH + "Data_Klaim.csv"
PATH_POLIS  = BASE_PATH + "Data_Polis.csv"
PATH_SAMPLE = BASE_PATH + "sample_submission.csv"

df_klaim  = pd.read_csv(PATH_KLAIM)
df_polis  = pd.read_csv(PATH_POLIS)
sample    = pd.read_csv(PATH_SAMPLE)

In [3]:
print("Klaim shape :", df_klaim.shape)

Klaim shape : (4627, 13)


In [4]:
print("Polis shape :", df_polis.shape)

Polis shape : (4096, 6)


In [5]:
print("Sample shape:", sample.shape)

Sample shape: (15, 2)


In [6]:
df_klaim.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4627 entries, 0 to 4626
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Claim ID                       4627 non-null   object 
 1   Nomor Polis                    4627 non-null   object 
 2   Reimburse/Cashless             4627 non-null   object 
 3   Inpatient/Outpatient           4590 non-null   object 
 4   ICD Diagnosis                  4621 non-null   object 
 5   ICD Description                4621 non-null   object 
 6   Status Klaim                   4627 non-null   object 
 7   Tanggal Pembayaran Klaim       4590 non-null   object 
 8   Tanggal Pasien Masuk RS        4627 non-null   object 
 9   Tanggal Pasien Keluar RS       4627 non-null   object 
 10  Nominal Klaim Yang Disetujui   4627 non-null   float64
 11  Nominal Biaya RS Yang Terjadi  4627 non-null   float64
 12  Lokasi RS                      4620 non-null   o

In [7]:
df_polis.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4096 entries, 0 to 4095
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Nomor Polis            4096 non-null   object
 1   Plan Code              4096 non-null   object
 2   Gender                 4096 non-null   object
 3   Tanggal Lahir          4096 non-null   int64 
 4   Tanggal Efektif Polis  4096 non-null   int64 
 5   Domisili               4096 non-null   object
dtypes: int64(2), object(4)
memory usage: 192.1+ KB


In [8]:
missing_klaim = (df_klaim.isna().mean() * 100).sort_values(ascending=False)
display(missing_klaim)

Inpatient/Outpatient             0.799654
Tanggal Pembayaran Klaim         0.799654
Lokasi RS                        0.151286
ICD Diagnosis                    0.129674
ICD Description                  0.129674
Nomor Polis                      0.000000
Reimburse/Cashless               0.000000
Claim ID                         0.000000
Status Klaim                     0.000000
Tanggal Pasien Masuk RS          0.000000
Tanggal Pasien Keluar RS         0.000000
Nominal Klaim Yang Disetujui     0.000000
Nominal Biaya RS Yang Terjadi    0.000000
dtype: float64

* Hampir semua data lengkap
* Missing di pembayaran 0.8 persen

In [9]:
missing_polis = (df_polis.isna().mean() * 100).sort_values(ascending=False)
display(missing_polis)

Nomor Polis              0.0
Plan Code                0.0
Gender                   0.0
Tanggal Lahir            0.0
Tanggal Efektif Polis    0.0
Domisili                 0.0
dtype: float64

duplikat

In [10]:
print("Duplicate Claim ID:", df_klaim["Claim ID"].duplicated().sum())
print("Duplicate Nomor Polis (Polis table):",
      df_polis["Nomor Polis"].duplicated().sum())

Duplicate Claim ID: 0
Duplicate Nomor Polis (Polis table): 0


In [11]:
display(df_klaim[[
    "Nominal Klaim Yang Disetujui",
    "Nominal Biaya RS Yang Terjadi"
]].describe().T)

,count,mean,std,min,25%,50%,75%,max
Nominal Klaim Yang Disetujui,4627.0,5.502892e+07,1.319527e+08,0.0,2274009.3,14467899.0,5.107209e+07,2.197500e+09
Nominal Biaya RS Yang Terjadi,4627.0,5.994940e+07,1.597838e+08,0.0,2720209.5,15871000.0,5.423067e+07,3.892810e+09


distribusi kategori

In [12]:
display(df_klaim["Status Klaim"].value_counts())
display(df_klaim["Reimburse/Cashless"].value_counts())
display(df_klaim["Inpatient/Outpatient"].value_counts())
display(df_polis["Plan Code"].value_counts())

Status Klaim
PAID    4627
Name: count, dtype: int64

Reimburse/Cashless
R    2722
C    1905
Name: count, dtype: int64

Inpatient/Outpatient
IP     2258
OP     1940
ODC     281
ODS     111
Name: count, dtype: int64

Plan Code
M-002    2253
M-003    1273
M-001     570
Name: count, dtype: int64

rentang tanggal 

In [13]:
# Buat kolom datetime versi baru (aman)
df_klaim["Tanggal Masuk_dt"] = pd.to_datetime(
    df_klaim["Tanggal Pasien Masuk RS"],
    errors="coerce"
)

df_klaim["Tanggal Bayar_dt"] = pd.to_datetime(
    df_klaim["Tanggal Pembayaran Klaim"],
    errors="coerce"
)

print("Convert selesai.")

Convert selesai.


In [14]:
print("RENTANG TANGGAL MASUK RS")
print("Min :", df_klaim["Tanggal Masuk_dt"].min())
print("Max :", df_klaim["Tanggal Masuk_dt"].max())

print("\RENTANG TANGGAL PEMBAYARAN")
print("Min :", df_klaim["Tanggal Bayar_dt"].min())
print("Max :", df_klaim["Tanggal Bayar_dt"].max())

RENTANG TANGGAL MASUK RS
Min : 2024-01-01 00:00:00
Max : 2025-07-31 00:00:00
\RENTANG TANGGAL PEMBAYARAN
Min : 2024-01-17 00:00:00
Max : 2025-12-09 00:00:00


<>:5: SyntaxWarning: invalid escape sequence '\R'
<>:5: SyntaxWarning: invalid escape sequence '\R'
/tmp/ipykernel_17/3160473776.py:5: SyntaxWarning: invalid escape sequence '\R'
  print("\RENTANG TANGGAL PEMBAYARAN")


# CLEANING STRING & FORMAT
Tahap Cleaning String & Format dilakukan untuk menstandarkan seluruh kolom bertipe teks agar memiliki format yang konsisten dan seragam. Seluruh nilai diubah menjadi string, spasi berlebih di awal, akhir, dan di tengah teks dibersihkan, serta huruf dikonversi menjadi uppercase. Proses ini diterapkan pada kolom-kolom kategorikal di data klaim dan data polis guna menghindari perbedaan penulisan yang sebenarnya merepresentasikan kategori yang sama, sehingga data menjadi lebih rapi, konsisten, dan siap digunakan untuk analisis lanjutan.

In [15]:
def clean_str(s):
    return (s.astype(str)
              .str.strip()
              .str.replace(r"\s+", " ", regex=True)
              .str.upper())

# --- Cleaning Klaim ---
str_cols_klaim = [
    "Claim ID","Nomor Polis","Reimburse/Cashless",
    "Inpatient/Outpatient","ICD Diagnosis",
    "ICD Description","Status Klaim","Lokasi RS"
]

for c in str_cols_klaim:
    if c in df_klaim.columns:
        df_klaim[c] = clean_str(df_klaim[c])

# --- Cleaning Polis ---
str_cols_polis = ["Nomor Polis","Plan Code","Gender","Domisili"]

for c in str_cols_polis:
    if c in df_polis.columns:
        df_polis[c] = clean_str(df_polis[c])

# PARSE TANGGAL & NUMERIK
Tahap Parse Tanggal & Numerik dilakukan untuk mengubah kolom tanggal dan nominal ke tipe data yang sesuai agar dapat dianalisis secara akurat. Kolom-kolom tanggal pada data klaim dan data polis dikonversi menjadi tipe datetime sehingga dapat digunakan untuk analisis berbasis waktu seperti agregasi bulanan dan perhitungan selisih hari. Parameter errors="coerce" memastikan bahwa nilai yang tidak valid akan diubah menjadi NaT tanpa menyebabkan error. Selanjutnya, kolom nominal klaim dibersihkan dari kemungkinan pemisah ribuan dan dikonversi menjadi tipe numerik (float), sehingga siap digunakan untuk perhitungan statistik dan agregasi seperti total klaim maupun severity.

In [16]:
date_cols_klaim = [
    "Tanggal Pembayaran Klaim",
    "Tanggal Pasien Masuk RS",
    "Tanggal Pasien Keluar RS"
]

for c in date_cols_klaim:
    df_klaim[c] = pd.to_datetime(df_klaim[c], errors="coerce")

date_cols_polis = ["Tanggal Lahir","Tanggal Efektif Polis"]
for c in date_cols_polis:
    df_polis[c] = pd.to_datetime(df_polis[c], errors="coerce")

# Numeric
num_cols = ["Nominal Klaim Yang Disetujui","Nominal Biaya RS Yang Terjadi"]
for c in num_cols:
    df_klaim[c] = (
        df_klaim[c]
        .astype(str)
        .str.replace(",", "", regex=False)
    )
    df_klaim[c] = pd.to_numeric(df_klaim[c], errors="coerce")

# DEDUP & FILTER STATUS
Tahap Dedup & Filter Status dilakukan untuk memastikan tidak terjadi perhitungan ganda dan hanya klaim yang valid yang digunakan dalam analisis. Data terlebih dahulu diurutkan berdasarkan Claim ID dan Tanggal Pembayaran Klaim, kemudian duplikasi pada Claim ID dihapus dengan mempertahankan entri terakhir. Setelah itu, data difilter agar hanya mencakup klaim dengan status PAID, sehingga yang dianalisis benar-benar klaim yang telah dibayarkan. Terakhir, baris dengan nilai nominal yang kosong atau bernilai negatif dihapus untuk menjaga konsistensi dan akurasi dalam perhitungan agregasi seperti total klaim dan severity.

In [17]:
# Deduplicate Claim ID
df_klaim = df_klaim.sort_values(
    ["Claim ID","Tanggal Pembayaran Klaim"]
)
df_klaim = df_klaim.drop_duplicates("Claim ID", keep="last")

# Filter hanya klaim PAID
df_klaim = df_klaim[df_klaim["Status Klaim"] == "PAID"].copy()

# Buang nominal negatif / NaN
df_klaim = df_klaim[df_klaim["Nominal Klaim Yang Disetujui"].notna()]
df_klaim = df_klaim[df_klaim["Nominal Klaim Yang Disetujui"] >= 0]

print("After filter:", df_klaim.shape)

After filter: (4627, 15)


# JOIN KLAIM & POLIS
Tahap Join Klaim & Polis dilakukan untuk menggabungkan data transaksi klaim dengan informasi profil pemegang polis berdasarkan kolom Nomor Polis. Proses penggabungan menggunakan metode left join, sehingga seluruh data klaim tetap dipertahankan meskipun terdapat kemungkinan data polis yang tidak cocok. Parameter validate="m:1" memastikan bahwa setiap klaim terhubung ke maksimal satu data polis, sehingga struktur relasi tetap konsisten. Setelah penggabungan, dilakukan pengecekan proporsi nilai kosong pada kolom seperti Plan Code untuk memastikan bahwa proses join berhasil dan tidak terdapat kehilangan informasi yang signifikan.

In [18]:
df = df_klaim.merge(
    df_polis,
    on="Nomor Polis",
    how="left",
    validate="m:1"
)

print("Missing polis info:", df["Plan Code"].isna().mean())

Missing polis info: 0.0


# FEATURE ENGINEERING DASAR
Tahap Feature Engineering Dasar dilakukan untuk menambahkan variabel turunan yang lebih informatif dari data mentah. Usia tertanggung saat kejadian klaim dihitung dari selisih tanggal masuk rumah sakit dengan tanggal lahir, sehingga dapat merepresentasikan profil risiko berdasarkan umur. Lama rawat inap (length of stay) dihitung dari selisih tanggal keluar dan masuk rumah sakit, dengan nilai negatif diubah menjadi kosong untuk menjaga konsistensi data. Selain itu, dibuat variabel biner untuk membedakan jenis perawatan (inpatient) dan metode klaim (cashless) agar mudah digunakan dalam analisis kuantitatif. Kode diagnosis ICD juga dikelompokkan menjadi tiga karakter pertama dan huruf awal untuk mengurangi tingkat granularitas, serta lokasi rumah sakit diklasifikasikan menjadi domestik dan internasional guna menangkap perbedaan pola klaim berdasarkan wilayah.

In [19]:
# Age at event
df["age_at_event"] = (
    (df["Tanggal Pasien Masuk RS"] - df["Tanggal Lahir"])
    .dt.days / 365.25
)

# Length of stay
df["los_days"] = (
    df["Tanggal Pasien Keluar RS"] -
    df["Tanggal Pasien Masuk RS"]
).dt.days

df.loc[df["los_days"] < 0, "los_days"] = np.nan

# Binary flags
df["is_inpatient"] = (df["Inpatient/Outpatient"] == "INPATIENT").astype(int)
df["is_cashless"]  = (df["Reimburse/Cashless"] == "CASHLESS").astype(int)

# ICD grouping
df["icd_3"] = df["ICD Diagnosis"].str[:3]
df["icd_chapter"] = df["ICD Diagnosis"].str[:1]

# Location grouping
df["loc_group"] = np.where(
    df["Lokasi RS"].str.contains("INDONESIA", na=False),
    "DOMESTIC",
    "INTERNATIONAL"
)

# BUAT DUAL TIMELINE
Tahap Buat Dual Timeline dilakukan untuk membuat dua acuan waktu berbeda dalam analisis, yaitu berdasarkan tanggal kejadian medis (Tanggal Pasien Masuk RS) dan tanggal pembayaran klaim (Tanggal Pembayaran Klaim). Kedua kolom tersebut diubah menjadi format periode bulanan (to_period("M")) agar memudahkan agregasi dan analisis time-series per bulan. Dengan pendekatan ini, tersedia dua perspektif waktu yang fleksibel, sehingga dapat dibandingkan apakah pola klaim lebih relevan dianalisis berdasarkan bulan kejadian atau bulan pembayaran. Pemeriksaan rentang periode juga membantu memastikan cakupan waktu data dan mengidentifikasi kemungkinan perbedaan horizon antara kejadian medis dan proses pembayaran.

In [20]:
df["ym_event"] = df["Tanggal Pasien Masuk RS"].dt.to_period("M")
df["ym_pay"]   = df["Tanggal Pembayaran Klaim"].dt.to_period("M")

print("Event range:", df["ym_event"].min(), "->", df["ym_event"].max())
print("Pay range  :", df["ym_pay"].min(), "->", df["ym_pay"].max())

Event range: 2024-01 -> 2025-07
Pay range  : 2024-01 -> 2025-12


# BUILD MONTHLY TABLE FUNCTION
Tahap Build Monthly Table Function bertujuan untuk mengubah data klaim level transaksi menjadi data agregasi bulanan yang siap digunakan untuk analisis time-series atau pemodelan. Fungsi build_monthly() mengelompokkan data berdasarkan kolom periode bulanan yang dipilih, kemudian menghitung berbagai metrik penting seperti jumlah klaim (frequency), total nilai klaim (total_claim), jumlah polis unik, proporsi rawat inap dan cashless, proporsi klaim internasional, serta rata-rata usia dan lama rawat. Setelah agregasi dilakukan, dihitung pula severity sebagai rasio antara total klaim dan jumlah klaim per bulan. Hasilnya adalah tabel bulanan yang lebih ringkas, informatif, dan fleksibel untuk digunakan baik dalam pendekatan berbasis kejadian medis (event) maupun pembayaran (pay).

In [21]:
def build_monthly(data, ym_col):
    m = (
        data.dropna(subset=[ym_col])
            .groupby(ym_col)
            .agg(
                frequency=("Claim ID","count"),
                total_claim=("Nominal Klaim Yang Disetujui","sum"),
                uniq_polis=("Nomor Polis","nunique"),
                prop_inpatient=("is_inpatient","mean"),
                prop_cashless=("is_cashless","mean"),
                share_international=("loc_group", lambda x: (x=="INTERNATIONAL").mean()),
                avg_age=("age_at_event","mean"),
                avg_los=("los_days","mean"),
            )
            .reset_index()
            .rename(columns={ym_col:"year_month"})
            .sort_values("year_month")
    )
    
    m["severity"] = m["total_claim"] / m["frequency"]
    m["severity"] = m["severity"].fillna(0)
    
    return m

monthly_event = build_monthly(df, "ym_event")
monthly_pay   = build_monthly(df, "ym_pay")

In [22]:
display(monthly_event.head(10))

,year_month,frequency,total_claim,uniq_polis,prop_inpatient,prop_cashless,share_international,avg_age,avg_los,severity
0,2024-01,302,2.026098e+10,179,0.0,0.0,0.317881,54.035864,1.609272,6.708934e+07
1,2024-02,208,1.385965e+10,120,0.0,0.0,0.326923,54.124901,1.557692,6.663291e+07
2,2024-03,278,1.431126e+10,174,0.0,0.0,0.338129,54.199499,1.503597,5.147935e+07
3,2024-04,239,1.144106e+10,138,0.0,0.0,0.309623,54.284696,1.188285,4.787056e+07
4,2024-05,263,1.221146e+10,152,0.0,0.0,0.292776,54.371874,1.193916,4.643141e+07
5,2024-06,225,1.212517e+10,130,0.0,0.0,0.302222,54.455511,1.368889,5.388963e+07
6,2024-07,257,1.497052e+10,142,0.0,0.0,0.346304,54.532459,0.953307,5.825104e+07
7,2024-08,228,1.351294e+10,138,0.0,0.0,0.280702,54.619006,1.535088,5.926726e+07
8,2024-09,208,1.226412e+10,121,0.0,0.0,0.307692,54.706747,1.427885,5.896211e+07
9,2024-10,274,1.268117e+10,158,0.0,0.0,0.368613,54.784714,1.098540,4.628163e+07


In [23]:
display(monthly_pay.head(10))

,year_month,frequency,total_claim,uniq_polis,prop_inpatient,prop_cashless,share_international,avg_age,avg_los,severity
0,2024-01,8,1.283162e+08,7,0.0,0.0,0.125000,54.021218,1.000000,1.603952e+07
1,2024-02,92,2.684171e+09,63,0.0,0.0,0.228261,54.034937,0.836957,2.917577e+07
2,2024-03,97,3.809944e+09,57,0.0,0.0,0.319588,54.082037,0.876289,3.927778e+07
3,2024-04,221,9.281203e+09,131,0.0,0.0,0.298643,54.102855,1.036199,4.199640e+07
4,2024-05,233,1.103847e+10,144,0.0,0.0,0.321888,54.190727,1.158798,4.737540e+07
5,2024-06,221,1.127720e+10,126,0.0,0.0,0.289593,54.247788,1.389140,5.102806e+07
6,2024-07,205,1.159773e+10,123,0.0,0.0,0.282927,54.301103,1.585366,5.657430e+07
7,2024-08,285,1.895989e+10,171,0.0,0.0,0.322807,54.386647,1.564912,6.652593e+07
8,2024-09,250,1.484250e+10,152,0.0,0.0,0.248000,54.484567,1.480000,5.937000e+07
9,2024-10,242,1.114198e+10,147,0.0,0.0,0.289256,54.573478,1.128099,4.604125e+07


# CEK COVERAGE FUTURE MONTH
Tahap Cek Coverage Future Month dilakukan untuk memastikan apakah bulan-bulan yang diminta pada sample submission sudah tersedia dalam tabel agregasi bulanan, baik berdasarkan timeline kejadian medis maupun pembayaran. Pertama, bulan target diekstrak dari kolom id pada sample submission dan diubah menjadi format periode bulanan. Selanjutnya, dibandingkan apakah setiap bulan target tersebut terdapat dalam monthly_pay dan monthly_event. Hasil pengecekan ini membantu menentukan apakah data masa depan sudah tercakup pada timeline pembayaran atau tidak, sehingga dapat mengindikasikan apakah strategi agregasi langsung memungkinkan atau perlu dilakukan forecasting murni berdasarkan kejadian medis.

In [24]:
sample["year"]  = sample["id"].str.split("_").str[0]
sample["month"] = sample["id"].str.split("_").str[1]

future_months = pd.PeriodIndex(
    sample["year"] + "-" + sample["month"],
    freq="M"
).unique()

pay_set = set(monthly_pay["year_month"].astype(str))
event_set = set(monthly_event["year_month"].astype(str))

print("Future months:", future_months)
print("PAY covers future:",
      [str(p) in pay_set for p in future_months])
print("EVENT covers future:",
      [str(p) in event_set for p in future_months])

Future months: PeriodIndex(['2025-08', '2025-09', '2025-10', '2025-11', '2025-12'], dtype='period[M]')
PAY covers future: [True, True, True, True, True]
EVENT covers future: [False, False, False, False, False]


# SIMPAN HASIL PREPROCESSING

In [25]:
OUT_DIR = "/kaggle/working/"

df.to_parquet(OUT_DIR + "claims_preprocessed.parquet", index=False)
monthly_event.to_parquet(OUT_DIR + "monthly_event.parquet", index=False)
monthly_pay.to_parquet(OUT_DIR + "monthly_pay.parquet", index=False)


# OUTPUT

Claim-level

In [26]:
print("CLAIM-LEVEL (df)")
print("Shape:", df.shape)
display(df.head(10))   

CLAIM-LEVEL (df)
Shape: (4627, 29)


,Claim ID,Nomor Polis,Reimburse/Cashless,Inpatient/Outpatient,ICD Diagnosis,ICD Description,Status Klaim,Tanggal Pembayaran Klaim,Tanggal Pasien Masuk RS,Tanggal Pasien Keluar RS,...,Domisili,age_at_event,los_days,is_inpatient,is_cashless,icd_3,icd_chapter,loc_group,ym_event,ym_pay
0,C-0001-M,POL-0176,R,OP,C50,MALIGNANT NEOPLASM OF BREAST,PAID,2024-07-08,2024-05-27,2024-05-27,...,JAKARTA,54.398357,0.0,0,0,C50,C,INTERNATIONAL,2024-05,2024-07
1,C-0002-M,POL-3288,R,OP,C34,MALIGNANT NEOPLASM OF BRONCHUS AND LUNG,PAID,2024-08-06,2024-07-15,2024-07-15,...,YOGYAKARTA,54.532512,0.0,0,0,C34,C,INTERNATIONAL,2024-07,2024-08
2,C-0003-M,POL-1786,R,OP,C18.9,"MALIGNANT NEOPLASM, COLON, UNSPECIFIED",PAID,2024-10-17,2024-05-16,2024-05-16,...,SURABAYA,54.368241,0.0,0,0,C18,C,INTERNATIONAL,2024-05,2024-10
3,C-0004-M,POL-1786,R,OP,C34,MALIGNANT NEOPLASM OF BRONCHUS AND LUNG,PAID,2024-09-03,2024-07-18,2024-07-18,...,SURABAYA,54.540726,0.0,0,0,C34,C,INTERNATIONAL,2024-07,2024-09
4,C-0005-M,POL-2778,R,OP,C50,MALIGNANT NEOPLASM OF BREAST,PAID,NaT,2024-06-06,2024-06-06,...,JAKARTA,54.425736,0.0,0,0,C50,C,INTERNATIONAL,2024-06,NaT
5,C-0006-M,POL-2778,R,OP,C50,MALIGNANT NEOPLASM OF BREAST,PAID,NaT,2024-07-18,2024-07-18,...,JAKARTA,54.540726,0.0,0,0,C50,C,INTERNATIONAL,2024-07,NaT
6,C-0007-M,POL-2778,R,OP,C50,MALIGNANT NEOPLASM OF BREAST,PAID,NaT,2024-06-27,2024-06-27,...,JAKARTA,54.483231,0.0,0,0,C50,C,INTERNATIONAL,2024-06,NaT
7,C-0008-M,POL-0566,R,OP,C90.0,MULTIPLE MYELOMA,PAID,NaT,2024-06-19,2024-06-19,...,JAKARTA,54.461328,0.0,0,0,C90,C,INTERNATIONAL,2024-06,NaT
8,C-0009-M,POL-2778,R,OP,C50,MALIGNANT NEOPLASM OF BREAST,PAID,NaT,2024-05-15,2024-05-15,...,JAKARTA,54.365503,0.0,0,0,C50,C,INTERNATIONAL,2024-05,NaT
9,C-0010-M,POL-0964,R,ODC,C82.9,"FOLLICULAR NON-HODGKIN'S LYMPHOMA, UNSPECIFIED",PAID,NaT,2024-07-05,2024-07-05,...,SURABAYA,54.505133,0.0,0,0,C82,C,INTERNATIONAL,2024-07,NaT


In [27]:
display(df.tail(10))   

,Claim ID,Nomor Polis,Reimburse/Cashless,Inpatient/Outpatient,ICD Diagnosis,ICD Description,Status Klaim,Tanggal Pembayaran Klaim,Tanggal Pasien Masuk RS,Tanggal Pasien Keluar RS,...,Domisili,age_at_event,los_days,is_inpatient,is_cashless,icd_3,icd_chapter,loc_group,ym_event,ym_pay
4617,C-5772-M,POL-3288,C,IP,C34.1,"MALIGNANT NEOPLASM OF UPPER LOBE, BRONCHUS OR ...",PAID,2025-01-31,2024-11-29,2024-11-29,...,YOGYAKARTA,54.907598,0.0,0,0,C34,C,INTERNATIONAL,2024-11,2025-01
4618,C-5773-M,POL-3288,R,ODC,H26.9,"CATARACT, UNSPECIFIED",PAID,2025-03-14,2025-01-08,2025-01-08,...,YOGYAKARTA,55.017112,0.0,0,0,H26,H,INTERNATIONAL,2025-01,2025-03
4619,C-5774-M,POL-3288,R,OP,H26,OTHER CATARACT,PAID,2025-02-07,2025-01-09,2025-01-09,...,YOGYAKARTA,55.019849,0.0,0,0,H26,H,INTERNATIONAL,2025-01,2025-02
4620,C-5775-M,POL-3288,C,IP,C34,MALIGNANT NEOPLASM OF BRONCHUS AND LUNG,PAID,2025-05-05,2025-02-12,2025-02-12,...,YOGYAKARTA,55.112936,0.0,0,0,C34,C,INTERNATIONAL,2025-02,2025-05
4621,C-5776-M,POL-3288,R,OP,C34.9,"BRONCHUS OR LUNG, UNSPECIFIED",PAID,2025-06-04,2025-03-10,2025-03-10,...,YOGYAKARTA,55.184120,0.0,0,0,C34,C,INTERNATIONAL,2025-03,2025-06
4622,C-5777-M,POL-3288,C,IP,C34,MALIGNANT NEOPLASM OF BRONCHUS AND LUNG,PAID,2025-07-02,2025-03-11,2025-03-11,...,YOGYAKARTA,55.186858,0.0,0,0,C34,C,INTERNATIONAL,2025-03,2025-07
4623,C-5778-M,POL-3288,C,IP,C34,MALIGNANT NEOPLASM OF BRONCHUS AND LUNG,PAID,2025-07-08,2025-04-18,2025-04-18,...,YOGYAKARTA,55.290897,0.0,0,0,C34,C,INTERNATIONAL,2025-04,2025-07
4624,C-5779-M,POL-3288,C,IP,C34.1,"MALIGNANT NEOPLASM OF UPPER LOBE, BRONCHUS OR ...",PAID,NaT,2025-05-21,2025-05-21,...,YOGYAKARTA,55.381246,0.0,0,0,C34,C,INTERNATIONAL,2025-05,NaT
4625,C-5780-M,POL-3288,C,IP,C34.9,"MALIGNANT NEOPLASM OF BRONCHUS OR LUNG, UNSPEC...",PAID,NaT,2025-06-30,2025-06-30,...,YOGYAKARTA,55.490760,0.0,0,0,C34,C,INTERNATIONAL,2025-06,NaT
4626,C-5781-M,POL-3288,C,IP,C34.1,"MALIGNANT NEOPLASM OF UPPER LOBE, BRONCHUS OR ...",PAID,2025-03-12,2025-01-07,2025-01-07,...,YOGYAKARTA,55.014374,0.0,0,0,C34,C,INTERNATIONAL,2025-01,2025-03


Struktur kolom, tipe data dan memory

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4627 entries, 0 to 4626
Data columns (total 29 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Claim ID                       4627 non-null   object        
 1   Nomor Polis                    4627 non-null   object        
 2   Reimburse/Cashless             4627 non-null   object        
 3   Inpatient/Outpatient           4627 non-null   object        
 4   ICD Diagnosis                  4627 non-null   object        
 5   ICD Description                4627 non-null   object        
 6   Status Klaim                   4627 non-null   object        
 7   Tanggal Pembayaran Klaim       4590 non-null   datetime64[ns]
 8   Tanggal Pasien Masuk RS        4627 non-null   datetime64[ns]
 9   Tanggal Pasien Keluar RS       4627 non-null   datetime64[ns]
 10  Nominal Klaim Yang Disetujui   4627 non-null   float64       
 11  Nominal Biaya RS 

Angka pada monthly_event dan monthly_pay berbeda karena keduanya menggunakan dasar waktu yang berbeda. monthly_event mengelompokkan klaim berdasarkan tanggal pasien masuk rumah sakit (waktu kejadian medis), sedangkan monthly_pay mengelompokkan klaim berdasarkan tanggal pembayaran (waktu klaim dibayarkan). Karena pembayaran klaim sering terjadi beberapa waktu setelah kejadian medis, satu klaim bisa tercatat di bulan yang berbeda pada masing-masing timeline. Perbedaan waktu pencatatan inilah yang menyebabkan nilai frequency dan total klaim per bulan tidak sama antara keduanya.

In [29]:
display(df.describe(include=[np.number]).T)

,count,mean,std,min,25%,50%,75%,max
Nominal Klaim Yang Disetujui,4627.0,5.502892e+07,1.319527e+08,0.000000,2.274009e+06,1.446790e+07,5.107209e+07,2.197500e+09
Nominal Biaya RS Yang Terjadi,4627.0,5.994940e+07,1.597838e+08,0.000000,2.720210e+06,1.587100e+07,5.423067e+07,3.892810e+09
age_at_event,4627.0,5.477279e+01,4.615147e-01,53.995893,5.436961e+01,5.477344e+01,5.516496e+01,5.557563e+01
los_days,4627.0,1.264102e+00,2.930572e+00,0.000000,0.000000e+00,0.000000e+00,1.000000e+00,5.400000e+01
is_inpatient,4627.0,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
is_cashless,4627.0,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [30]:
display(df.describe(include=["object", "category"]).T)

,count,unique,top,freq
Claim ID,4627,4627,C-5781-M,1
Nomor Polis,4627,1210,POL-2078,222
Reimburse/Cashless,4627,2,R,2722
Inpatient/Outpatient,4627,5,IP,2258
ICD Diagnosis,4627,753,N18.0,290
ICD Description,4627,841,END-STAGE RENAL DISEASE,290
Status Klaim,4627,1,PAID,4627
Lokasi RS,4627,11,INDONESIA,3102
Plan Code,4627,3,M-002,3028
Gender,4627,2,F,2327


missing value

In [31]:
miss = (df.isna().mean().sort_values(ascending=False) * 100).to_frame("missing_%")
display(miss.head(30))

,missing_%
Tanggal Pembayaran Klaim,0.799654
ym_pay,0.799654
Tanggal Bayar_dt,0.799654
Nomor Polis,0.000000
Reimburse/Cashless,0.000000
ICD Diagnosis,0.000000
ICD Description,0.000000
Status Klaim,0.000000
Tanggal Pasien Masuk RS,0.000000
Tanggal Pasien Keluar RS,0.000000


dataset bulanan (event & payment)

In [32]:
print("Shape:", monthly_event.shape)
display(monthly_event.head(12))
display(monthly_event.tail(12))

Shape: (19, 10)


,year_month,frequency,total_claim,uniq_polis,prop_inpatient,prop_cashless,share_international,avg_age,avg_los,severity
0,2024-01,302,2.026098e+10,179,0.0,0.0,0.317881,54.035864,1.609272,6.708934e+07
1,2024-02,208,1.385965e+10,120,0.0,0.0,0.326923,54.124901,1.557692,6.663291e+07
2,2024-03,278,1.431126e+10,174,0.0,0.0,0.338129,54.199499,1.503597,5.147935e+07
3,2024-04,239,1.144106e+10,138,0.0,0.0,0.309623,54.284696,1.188285,4.787056e+07
4,2024-05,263,1.221146e+10,152,0.0,0.0,0.292776,54.371874,1.193916,4.643141e+07
5,2024-06,225,1.212517e+10,130,0.0,0.0,0.302222,54.455511,1.368889,5.388963e+07
6,2024-07,257,1.497052e+10,142,0.0,0.0,0.346304,54.532459,0.953307,5.825104e+07
7,2024-08,228,1.351294e+10,138,0.0,0.0,0.280702,54.619006,1.535088,5.926726e+07
8,2024-09,208,1.226412e+10,121,0.0,0.0,0.307692,54.706747,1.427885,5.896211e+07
9,2024-10,274,1.268117e+10,158,0.0,0.0,0.368613,54.784714,1.098540,4.628163e+07


,year_month,frequency,total_claim,uniq_polis,prop_inpatient,prop_cashless,share_international,avg_age,avg_los,severity
7,2024-08,228,1.351294e+10,138,0.0,0.0,0.280702,54.619006,1.535088,5.926726e+07
8,2024-09,208,1.226412e+10,121,0.0,0.0,0.307692,54.706747,1.427885,5.896211e+07
9,2024-10,274,1.268117e+10,158,0.0,0.0,0.368613,54.784714,1.098540,4.628163e+07
10,2024-11,270,1.373306e+10,147,0.0,0.0,0.388889,54.870028,1.077778,5.086318e+07
11,2024-12,238,1.201391e+10,133,0.0,0.0,0.319328,54.947262,1.197479,5.047861e+07
12,2025-01,216,9.610380e+09,132,0.0,0.0,0.282407,55.037988,1.212963,4.449250e+07
13,2025-02,246,1.748054e+10,145,0.0,0.0,0.341463,55.119837,1.239837,7.105911e+07
14,2025-03,230,1.367924e+10,126,0.0,0.0,0.400000,55.196738,1.078261,5.947496e+07
15,2025-04,208,1.116425e+10,123,0.0,0.0,0.336538,55.287290,1.201923,5.367427e+07
16,2025-05,239,1.222680e+10,142,0.0,0.0,0.297071,55.369321,1.301255,5.115814e+07


In [33]:
print("Shape:", monthly_pay.shape)
display(monthly_pay.head(12))
display(monthly_pay.tail(12))

Shape: (24, 10)


,year_month,frequency,total_claim,uniq_polis,prop_inpatient,prop_cashless,share_international,avg_age,avg_los,severity
0,2024-01,8,1.283162e+08,7,0.0,0.0,0.125000,54.021218,1.000000,1.603952e+07
1,2024-02,92,2.684171e+09,63,0.0,0.0,0.228261,54.034937,0.836957,2.917577e+07
2,2024-03,97,3.809944e+09,57,0.0,0.0,0.319588,54.082037,0.876289,3.927778e+07
3,2024-04,221,9.281203e+09,131,0.0,0.0,0.298643,54.102855,1.036199,4.199640e+07
4,2024-05,233,1.103847e+10,144,0.0,0.0,0.321888,54.190727,1.158798,4.737540e+07
5,2024-06,221,1.127720e+10,126,0.0,0.0,0.289593,54.247788,1.389140,5.102806e+07
6,2024-07,205,1.159773e+10,123,0.0,0.0,0.282927,54.301103,1.585366,5.657430e+07
7,2024-08,285,1.895989e+10,171,0.0,0.0,0.322807,54.386647,1.564912,6.652593e+07
8,2024-09,250,1.484250e+10,152,0.0,0.0,0.248000,54.484567,1.480000,5.937000e+07
9,2024-10,242,1.114198e+10,147,0.0,0.0,0.289256,54.573478,1.128099,4.604125e+07


,year_month,frequency,total_claim,uniq_polis,prop_inpatient,prop_cashless,share_international,avg_age,avg_los,severity
12,2025-01,293,1.697253e+10,163,0.0,0.0,0.348123,54.877360,1.180887,5.792671e+07
13,2025-02,183,9.559585e+09,123,0.0,0.0,0.409836,54.971182,0.786885,5.223817e+07
14,2025-03,234,1.494105e+10,147,0.0,0.0,0.320513,55.037926,1.700855,6.385066e+07
15,2025-04,184,7.538943e+09,109,0.0,0.0,0.315217,55.132444,1.092391,4.097252e+07
16,2025-05,201,9.628068e+09,130,0.0,0.0,0.184080,55.180020,1.666667,4.790084e+07
17,2025-06,204,1.617766e+10,122,0.0,0.0,0.490196,55.261921,0.877451,7.930225e+07
18,2025-07,272,1.862361e+10,150,0.0,0.0,0.341912,55.352589,1.257353,6.846917e+07
19,2025-08,245,1.546896e+10,134,0.0,0.0,0.346939,55.449603,1.216327,6.313861e+07
20,2025-09,197,1.041073e+10,124,0.0,0.0,0.355330,55.496208,1.294416,5.284633e+07
21,2025-10,58,4.900102e+09,43,0.0,0.0,0.258621,55.525833,1.982759,8.448453e+07


bulan target submission (Aug–Dec 2025)

In [34]:
# ambil bulan dari sample_submission
tmp = sample.copy()
tmp["year"]  = tmp["id"].str.split("_").str[0].astype(int)
tmp["month"] = tmp["id"].str.split("_").str[1].astype(int)
future_months = pd.PeriodIndex(tmp["year"].astype(str) + "-" + tmp["month"].astype(str).str.zfill(2), freq="M").unique().sort_values()

print("Future months:", list(future_months.astype(str)))

# tampilkan monthly_pay/monthly_event pada bulan target
print("MONTHLY PAY (future months only)")
display(monthly_pay[monthly_pay["year_month"].isin(future_months)])

print("MONTHLY EVENT (future months only)")
display(monthly_event[monthly_event["year_month"].isin(future_months)])

Future months: ['2025-08', '2025-09', '2025-10', '2025-11', '2025-12']
MONTHLY PAY (future months only)


,year_month,frequency,total_claim,uniq_polis,prop_inpatient,prop_cashless,share_international,avg_age,avg_los,severity
19,2025-08,245,1.546896e+10,134,0.0,0.0,0.346939,55.449603,1.216327,6.313861e+07
20,2025-09,197,1.041073e+10,124,0.0,0.0,0.355330,55.496208,1.294416,5.284633e+07
21,2025-10,58,4.900102e+09,43,0.0,0.0,0.258621,55.525833,1.982759,8.448453e+07
22,2025-11,3,1.356322e+08,3,0.0,0.0,0.333333,55.540041,3.000000,4.521074e+07
23,2025-12,2,1.366003e+08,2,0.0,0.0,1.000000,55.550992,3.500000,6.830014e+07


MONTHLY EVENT (future months only)


,year_month,frequency,total_claim,uniq_polis,prop_inpatient,prop_cashless,share_international,avg_age,avg_los,severity
